In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display

print("04 SBERT Semantic Recommendation - Started")

04 SBERT Semantic Recommendation - Started


In [2]:
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"

MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:")
print(PROJECT_DIR)

print("\nFeatures directory:")
print(FEATURES_DIR)

print("\nResults directory:")
print(RESULTS_DIR)

Project directory:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation

Features directory:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\data\features

Results directory:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results


In [3]:
destinations = pd.read_csv(
    PROCESSED_DIR / "destinations_clean.csv"
)

users = pd.read_csv(
    PROCESSED_DIR / "user_profiles_clean.csv"
)

evaluation_cases = pd.read_csv(
    PROCESSED_DIR / "evaluation_cases_clean.csv"
)

print("Datasets loaded successfully.")

print("\nUsers:", users.shape)
print("Destinations:", destinations.shape)
print("Evaluation cases:", evaluation_cases.shape)

Datasets loaded successfully.

Users: (50, 13)
Destinations: (100, 21)
Evaluation cases: (500, 27)


In [4]:
destination_text = pd.read_csv(
    FEATURES_DIR / "destination_text.csv"
)

user_text = pd.read_csv(
    FEATURES_DIR / "user_text.csv"
)

print("Destination text:")
print(destination_text.shape)

print("\nUser text:")
print(user_text.shape)

print("\nDestination columns:")
print(destination_text.columns.tolist())

print("\nUser columns:")
print(user_text.columns.tolist())

Destination text:
(100, 3)

User text:
(50, 2)

Destination columns:
['destination_id', 'name', 'destination_text']

User columns:
['user_id', 'user_preference_text']


In [5]:
print("========== DESTINATION TEXT ==========")

display(
    destination_text.head(10)
)

print("\n========== USER PREFERENCE TEXT ==========")

display(
    user_text.head(10)
)

========== DESTINATION TEXT ==========


,destination_id,name,destination_text
0,D001,Sigiriya Rock Fortress,sigiriya rock fortress matale heritage nature ...
1,D002,Dambulla Cave Temple,dambulla cave temple matale heritage heritage ...
2,D003,Minneriya National Park,minneriya national park polonnaruwa wildlife n...
3,D004,Kaudulla National Park,kaudulla national park polonnaruwa wildlife na...
4,D005,Polonnaruwa Ancient City,polonnaruwa ancient city polonnaruwa heritage ...
5,D006,Anuradhapura Sacred City,anuradhapura sacred city anuradhapura heritage...
6,D007,Ritigala Monastery,ritigala monastery anuradhapura heritage natur...
7,D008,Mihintale,mihintale anuradhapura heritage religious heri...
8,D009,Wilpattu National Park,wilpattu national park puttalam wildlife natur...
9,D010,Jaffna Fort,jaffna fort jaffna heritage heritage history c...



========== USER PREFERENCE TEXT ==========


,user_id,user_preference_text
0,U001,nature photography budget solo low cool colombo
1,U002,beach relaxation moderate couple medium warm k...
2,U003,heritage history budget friends low dry gampaha
3,U004,wildlife photography moderate family medium mi...
4,U005,adventure hiking premium solo high rain_tolera...
5,U006,culture food budget couple low cool galle
6,U007,nature hiking moderate friends medium warm kur...
7,U008,beach adventure budget family low dry ratnapura
8,U009,heritage culture moderate solo medium mixed ba...
9,U010,wildlife nature premium couple high rain_toler...


In [6]:
print(
    "Missing destination text:",
    destination_text["destination_text"].isna().sum()
)

print(
    "Missing user preference text:",
    user_text["user_preference_text"].isna().sum()
)

Missing destination text: 0
Missing user preference text: 0


In [7]:
destination_text["destination_text"] = (
    destination_text["destination_text"]
    .fillna("")
    .astype(str)
)

user_text["user_preference_text"] = (
    user_text["user_preference_text"]
    .fillna("")
    .astype(str)
)

print("Text cleaning completed.")

Text cleaning completed.


In [8]:
MODEL_NAME = "all-MiniLM-L6-v2"

sbert_model = SentenceTransformer(
    MODEL_NAME
)

print("SBERT model loaded:")
print(MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SBERT model loaded:
all-MiniLM-L6-v2


In [9]:
destination_embeddings = sbert_model.encode(
    destination_text["destination_text"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print(
    "Destination embedding shape:",
    destination_embeddings.shape
)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Destination embedding shape: (100, 384)


In [10]:
user_embeddings = sbert_model.encode(
    user_text["user_preference_text"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print(
    "User embedding shape:",
    user_embeddings.shape
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

User embedding shape: (50, 384)


In [11]:
np.save(
    FEATURES_DIR / "sbert_destination_embeddings.npy",
    destination_embeddings
)

np.save(
    FEATURES_DIR / "sbert_user_embeddings.npy",
    user_embeddings
)

print("SBERT embeddings saved successfully.")

SBERT embeddings saved successfully.


In [12]:
semantic_similarity_matrix = cosine_similarity(
    user_embeddings,
    destination_embeddings
)

print(
    "Semantic similarity matrix:",
    semantic_similarity_matrix.shape
)

print(
    "Minimum similarity:",
    semantic_similarity_matrix.min()
)

print(
    "Maximum similarity:",
    semantic_similarity_matrix.max()
)

Semantic similarity matrix: (50, 100)
Minimum similarity: -0.02939562
Maximum similarity: 0.72475064


In [13]:
def sbert_recommend(
    user_id,
    top_k=10
):

    user_positions = user_text.index[
        user_text["user_id"] == user_id
    ]

    if len(user_positions) == 0:
        raise ValueError(
            f"User {user_id} not found."
        )

    user_position = user_positions[0]

    similarities = semantic_similarity_matrix[
        user_position
    ]

    result = destinations.copy()

    result["semantic_similarity"] = similarities

    result = result.sort_values(
        "semantic_similarity",
        ascending=False
    ).head(top_k).copy()

    result["user_id"] = user_id

    result["rank"] = range(
        1,
        len(result) + 1
    )

    return result[
        [
            "user_id",
            "destination_id",
            "name",
            "category",
            "district",
            "estimated_cost_lkr",
            "sustainability_score",
            "crowd_score",
            "seasonality_score",
            "weather_suitability",
            "semantic_similarity",
            "rank"
        ]
    ]

In [14]:
test_user_id = users["user_id"].iloc[0]

sbert_test = sbert_recommend(
    test_user_id,
    top_k=10
)

print(
    "Test user:",
    test_user_id
)

display(sbert_test)

Test user: U001


,user_id,destination_id,name,category,district,estimated_cost_lkr,sustainability_score,crowd_score,seasonality_score,weather_suitability,semantic_similarity,rank
50,U001,D051,Viharamahadevi Park,nature;city,colombo,2000,58,29,56,0.803,0.695178,1
51,U001,D052,Independence Square,heritage;city,colombo,2000,60,38,49,0.743,0.606650,2
62,U001,D063,Sinharaja Forest Reserve,nature;wildlife,ratnapura,8000,69,26,67,0.739,0.594293,3
48,U001,D049,Galle Face Green,city;coastal,colombo,2000,55,27,55,0.452,0.582409,4
82,U001,D083,Hummanaya Blow Hole,nature;coastal,matara,3000,58,34,67,0.827,0.581182,5
96,U001,D097,Haputale,nature;city,badulla,4000,61,28,52,0.836,0.580706,6
72,U001,D073,Kanneliya Forest Reserve,nature;wildlife,galle,7000,62,38,63,0.647,0.577611,7
31,U001,D032,Ravana Falls,nature,badulla,3000,65,33,59,0.896,0.575103,8
2,U001,D003,Minneriya National Park,wildlife;nature,polonnaruwa,15000,70,35,59,0.382,0.572962,9
63,U001,D064,Udawalawe National Park,wildlife;nature,ratnapura,15000,64,25,66,0.756,0.572826,10


In [15]:
print(
    "Semantic similarity range:"
)

print(
    "Minimum:",
    sbert_test["semantic_similarity"].min()
)

print(
    "Maximum:",
    sbert_test["semantic_similarity"].max()
)

print(
    "Mean:",
    sbert_test["semantic_similarity"].mean()
)

Semantic similarity range:
Minimum: 0.5728259
Maximum: 0.6951784
Mean: 0.593892


In [16]:
sbert_all = []

for user_id in users["user_id"]:

    result = sbert_recommend(
        user_id,
        top_k=10
    )

    sbert_all.append(result)

all_sbert_results = pd.concat(
    sbert_all,
    ignore_index=True
)

print(
    "All SBERT recommendations:",
    all_sbert_results.shape
)

display(
    all_sbert_results.head(10)
)

All SBERT recommendations: (500, 12)


,user_id,destination_id,name,category,district,estimated_cost_lkr,sustainability_score,crowd_score,seasonality_score,weather_suitability,semantic_similarity,rank
0,U001,D051,Viharamahadevi Park,nature;city,colombo,2000,58,29,56,0.803,0.695178,1
1,U001,D052,Independence Square,heritage;city,colombo,2000,60,38,49,0.743,0.606650,2
2,U001,D063,Sinharaja Forest Reserve,nature;wildlife,ratnapura,8000,69,26,67,0.739,0.594293,3
3,U001,D049,Galle Face Green,city;coastal,colombo,2000,55,27,55,0.452,0.582409,4
4,U001,D083,Hummanaya Blow Hole,nature;coastal,matara,3000,58,34,67,0.827,0.581182,5
5,U001,D097,Haputale,nature;city,badulla,4000,61,28,52,0.836,0.580706,6
6,U001,D073,Kanneliya Forest Reserve,nature;wildlife,galle,7000,62,38,63,0.647,0.577611,7
7,U001,D032,Ravana Falls,nature,badulla,3000,65,33,59,0.896,0.575103,8
8,U001,D003,Minneriya National Park,wildlife;nature,polonnaruwa,15000,70,35,59,0.382,0.572962,9
9,U001,D064,Udawalawe National Park,wildlife;nature,ratnapura,15000,64,25,66,0.756,0.572826,10


In [17]:
sbert_results_path = (
    RESULTS_DIR
    / "b5_sbert_semantic_all_users_top10.csv"
)

all_sbert_results.to_csv(
    sbert_results_path,
    index=False
)

print("SBERT recommendations saved:")
print(sbert_results_path)

SBERT recommendations saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results\b5_sbert_semantic_all_users_top10.csv


In [18]:
all_sbert_results[
    "semantic_similarity_normalized"
] = (
    all_sbert_results
    .groupby("user_id")[
        "semantic_similarity"
    ]
    .transform(
        lambda x:
        (
            (x - x.min())
            /
            (x.max() - x.min())
        )
        if x.max() != x.min()
        else 1.0
    )
)

print(
    "SBERT similarity normalized."
)

display(
    all_sbert_results.head()
)

SBERT similarity normalized.


,user_id,destination_id,name,category,district,estimated_cost_lkr,sustainability_score,crowd_score,seasonality_score,weather_suitability,semantic_similarity,rank,semantic_similarity_normalized
0,U001,D051,Viharamahadevi Park,nature;city,colombo,2000,58,29,56,0.803,0.695178,1,1.000000
1,U001,D052,Independence Square,heritage;city,colombo,2000,60,38,49,0.743,0.606650,2,0.276449
2,U001,D063,Sinharaja Forest Reserve,nature;wildlife,ratnapura,8000,69,26,67,0.739,0.594293,3,0.175449
3,U001,D049,Galle Face Green,city;coastal,colombo,2000,55,27,55,0.452,0.582409,4,0.078326
4,U001,D083,Hummanaya Blow Hole,nature;coastal,matara,3000,58,34,67,0.827,0.581182,5,0.068299


In [24]:
# Load the B4 hybrid recommendations
b4_path = RESULTS_DIR / "b4_hybrid_all_users_top10.csv"

hybrid_results = pd.read_csv(b4_path)

print("B4 hybrid results:", hybrid_results.shape)

display(hybrid_results.head())

B4 hybrid results: (206, 19)


,user_id,destination_id,content_similarity,content_similarity_normalized,context_score,hybrid_score,rank,name,category,district,estimated_cost_lkr,duration_hours,sustainability_score,crowd_score,seasonality_score,weather_temp_c,weather_rain_risk,weather_suitability,explanation
0,U002,D088,0.499895,1.000000,0.832527,0.933011,1,Tangalle Beach,beach,hambantota,5000,4.0,55,37,61,28.8,0.061,0.939,strong preference match; good contextual suita...
1,U002,D041,0.444962,0.700368,0.815013,0.746226,2,Pasikuda Beach,beach,batticaloa,7000,4.0,58,35,51,27.2,0.249,0.751,strong preference match; good contextual suita...
2,U002,D047,0.437344,0.658816,0.822867,0.724436,3,Marble Beach,beach,trincomalee,5000,3.0,52,33,61,28.9,0.259,0.741,strong preference match; good contextual suita...
3,U002,D012,0.428244,0.609184,0.857480,0.708502,4,Casuarina Beach,beach,jaffna,4000,3.0,58,43,50,29.1,0.134,0.866,strong preference match; good contextual suita...
4,U002,D079,0.404958,0.482170,0.808407,0.612665,5,Mirissa Beach,beach,matara,6000,4.0,56,36,63,24.2,0.301,0.699,good contextual suitability


In [20]:
required_b4_columns = [
    "user_id",
    "destination_id",
    "hybrid_score"
]

print(
    "Checking B4 columns..."
)

for column in required_b4_columns:

    print(
        column,
        "→",
        column in hybrid_results.columns
    )

Checking B4 columns...
user_id → True
destination_id → True
hybrid_score → True


In [21]:
sbert_for_hybrid = all_sbert_results[
    [
        "user_id",
        "destination_id",
        "semantic_similarity",
        "semantic_similarity_normalized"
    ]
].copy()

print(
    "SBERT hybrid data:",
    sbert_for_hybrid.shape
)

SBERT hybrid data: (500, 4)


In [22]:
sbert_hybrid = pd.merge(
    hybrid_results[
        [
            "user_id",
            "destination_id",
            "hybrid_score"
        ]
    ],

    sbert_for_hybrid,

    on=[
        "user_id",
        "destination_id"
    ],

    how="inner"
)

print(
    "Merged SBERT + hybrid:",
    sbert_hybrid.shape
)

display(
    sbert_hybrid.head(10)
)

Merged SBERT + hybrid: (113, 5)


,user_id,destination_id,hybrid_score,semantic_similarity,semantic_similarity_normalized
0,U002,D088,0.933011,0.543972,0.245419
1,U002,D041,0.746226,0.559832,0.448996
2,U002,D047,0.724436,0.544621,0.253747
3,U002,D012,0.708502,0.575921,0.655504
4,U002,D079,0.612665,0.524852,0.000000
5,U002,D075,0.338619,0.558384,0.430405
6,U003,D037,0.903435,0.391323,0.647600
7,U003,D010,0.443328,0.361271,0.142288
8,U003,D002,0.372944,0.362361,0.160612
9,U004,D059,0.518707,0.642971,0.773596


In [23]:
print(
    "Number of merged recommendations:",
    len(sbert_hybrid)
)

Number of merged recommendations: 113


In [25]:
print("B4 columns:")
print(hybrid_results.columns.tolist())

print("\nUsers in B4:")
print(hybrid_results["user_id"].nunique())

print("\nB4 rows:")
print(len(hybrid_results))

B4 columns:
['user_id', 'destination_id', 'content_similarity', 'content_similarity_normalized', 'context_score', 'hybrid_score', 'rank', 'name', 'category', 'district', 'estimated_cost_lkr', 'duration_hours', 'sustainability_score', 'crowd_score', 'seasonality_score', 'weather_temp_c', 'weather_rain_risk', 'weather_suitability', 'explanation']

Users in B4:
45

B4 rows:
206


In [30]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded.")

Libraries loaded.


In [31]:
print("Users:", users.shape)
print("Destinations:", destinations.shape)

Users: (50, 13)
Destinations: (100, 21)


In [32]:
print("User text columns:")
print(user_text.columns.tolist())

print("\nDestination text columns:")
print(destination_text.columns.tolist())

User text columns:
['user_id', 'user_preference_text']

Destination text columns:
['destination_id', 'name', 'destination_text']


In [33]:
print("User text columns:")
print(user_text.columns.tolist())

print("\nDestination text columns:")
print(destination_text.columns.tolist())

User text columns:
['user_id', 'user_preference_text']

Destination text columns:
['destination_id', 'name', 'destination_text']


In [34]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1
)

combined_text = pd.concat([
    user_text["user_preference_text"].fillna(""),
    destination_text["destination_text"].fillna("")
])

tfidf.fit(combined_text)

print(
    "TF-IDF vocabulary size:",
    len(tfidf.vocabulary_)
)

TF-IDF vocabulary size: 647


In [35]:
tfidf_user_vectors = tfidf.transform(
    user_text["user_preference_text"].fillna("")
)

tfidf_destination_vectors = tfidf.transform(
    destination_text["destination_text"].fillna("")
)

tfidf_similarity_matrix = cosine_similarity(
    tfidf_user_vectors,
    tfidf_destination_vectors
)

print(
    "TF-IDF matrix shape:",
    tfidf_similarity_matrix.shape
)

TF-IDF matrix shape: (50, 100)


In [36]:
tfidf_score_matrix = pd.DataFrame(
    tfidf_similarity_matrix,
    index=users["user_id"],
    columns=destinations["destination_id"]
)

tfidf_long = (
    tfidf_score_matrix
    .reset_index()
    .melt(
        id_vars="user_id",
        var_name="destination_id",
        value_name="content_similarity"
    )
)

print(
    "TF-IDF pairs:",
    tfidf_long.shape
)

TF-IDF pairs: (5000, 3)


In [37]:
tfidf_long["content_similarity_normalized"] = (
    tfidf_long
    .groupby("user_id")["content_similarity"]
    .transform(
        lambda x:
        (x - x.min()) /
        (x.max() - x.min())
        if x.max() != x.min()
        else 1.0
    )
)

print("TF-IDF normalization completed.")

TF-IDF normalization completed.


In [39]:
# Cell 8 — Build context scores for all 50 users × 100 destinations

import numpy as np
import pandas as pd


def parse_interests(value):
    if pd.isna(value):
        return []
    
    return [
        x.strip().lower()
        for x in str(value).split(";")
        if x.strip()
    ]


def interest_match(user_interests, destination_interests):
    user_set = set(parse_interests(user_interests))
    destination_set = set(parse_interests(destination_interests))

    if not user_set or not destination_set:
        return 0.0

    return len(user_set.intersection(destination_set)) / len(user_set)


def calculate_budget_fit(user_budget, destination_cost):
    if pd.isna(user_budget) or pd.isna(destination_cost):
        return 0.0

    user_budget = float(user_budget)
    destination_cost = float(destination_cost)

    if destination_cost <= user_budget:
        return 1.0

    ratio = user_budget / destination_cost

    return max(0.0, min(1.0, ratio))


def calculate_crowd_fit(preferred_crowd, destination_crowd):
    if pd.isna(preferred_crowd) or pd.isna(destination_crowd):
        return 0.0

    preferred = str(preferred_crowd).lower().strip()
    crowd = float(destination_crowd)

    if preferred == "low":
        return max(0.0, min(1.0, (70 - crowd) / 70))

    elif preferred == "medium":
        return max(0.0, min(1.0, 1 - abs(crowd - 50) / 50))

    elif preferred == "high":
        return max(0.0, min(1.0, crowd / 70))

    return 0.0


def calculate_weather_fit(preferred_weather, weather_suitability):
    if pd.isna(weather_suitability):
        return 0.0

    suitability = float(weather_suitability)

    if suitability > 1:
        suitability = suitability / 100

    return max(0.0, min(1.0, suitability))


context_rows = []

for _, user in users.iterrows():

    for _, destination in destinations.iterrows():

        preference_score = interest_match(
            user["preferred_interests"],
            destination["interest_tags"]
        )

        budget_score = calculate_budget_fit(
            user["budget_lkr_per_day"],
            destination["estimated_cost_lkr"]
        )

        crowd_fit_score = calculate_crowd_fit(
            user["preferred_crowd"],
            destination["crowd_score"]
        )

        weather_fit_score = calculate_weather_fit(
            user["preferred_weather"],
            destination["weather_suitability"]
        )

        # Context score
        context_score = (
            0.45 * preference_score +
            0.25 * budget_score +
            0.15 * crowd_fit_score +
            0.15 * weather_fit_score
        )

        context_rows.append({
            "user_id": user["user_id"],
            "destination_id": destination["destination_id"],
            "preference_score": preference_score,
            "budget_score": budget_score,
            "crowd_fit_score": crowd_fit_score,
            "weather_fit_score": weather_fit_score,
            "context_score": context_score
        })


all_context_scores = pd.DataFrame(context_rows)

print("all_context_scores created successfully")
print("Shape:", all_context_scores.shape)

display(all_context_scores.head())

all_context_scores created successfully
Shape: (5000, 7)


,user_id,destination_id,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score
0,U001,D001,0.5,0.416667,0.414286,0.516,0.468710
1,U001,D002,0.5,1.000000,0.414286,0.754,0.650243
2,U001,D003,1.0,0.333333,0.500000,0.382,0.665633
3,U001,D004,1.0,0.333333,0.528571,0.366,0.667519
4,U001,D005,0.5,1.000000,0.542857,0.467,0.626479


In [40]:
# Cell 9 — Check context scores

print("all_context_scores shape:", all_context_scores.shape)

print("\nColumns:")
print(all_context_scores.columns.tolist())

print("\nUsers:")
print(all_context_scores["user_id"].nunique())

print("\nDestinations:")
print(all_context_scores["destination_id"].nunique())

print("\nContext score range:")
print(
    all_context_scores["context_score"].min(),
    "to",
    all_context_scores["context_score"].max()
)

display(all_context_scores.head(10))

all_context_scores shape: (5000, 7)

Columns:
['user_id', 'destination_id', 'preference_score', 'budget_score', 'crowd_fit_score', 'weather_fit_score', 'context_score']

Users:
50

Destinations:
100

Context score range:
0.21046785714285712 to 0.98395


,user_id,destination_id,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score
0,U001,D001,0.5,0.416667,0.414286,0.516,0.468710
1,U001,D002,0.5,1.000000,0.414286,0.754,0.650243
2,U001,D003,1.0,0.333333,0.500000,0.382,0.665633
3,U001,D004,1.0,0.333333,0.528571,0.366,0.667519
4,U001,D005,0.5,1.000000,0.542857,0.467,0.626479
5,U001,D006,0.5,1.000000,0.585714,0.645,0.659607
6,U001,D007,0.5,1.000000,0.585714,0.436,0.628257
7,U001,D008,0.5,1.000000,0.471429,0.808,0.666914
8,U001,D009,1.0,0.312500,0.600000,0.827,0.742175
9,U001,D010,0.5,1.000000,0.571429,0.942,0.702014


In [41]:
# Cell 10 — Merge content similarity with context scores

all_hybrid = tfidf_long.merge(
    all_context_scores,
    on=["user_id", "destination_id"],
    how="inner"
)

print("all_hybrid shape:", all_hybrid.shape)

display(all_hybrid.head())

all_hybrid shape: (5000, 9)


,user_id,destination_id,content_similarity,content_similarity_normalized,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score
0,U001,D001,0.021762,0.223653,0.5,0.416667,0.414286,0.516,0.468710
1,U002,D001,0.000000,0.000000,0.0,0.666667,0.820000,0.516,0.367067
2,U003,D001,0.124593,0.760817,1.0,0.916667,0.414286,0.516,0.818710
3,U004,D001,0.009276,0.072408,0.5,1.000000,0.820000,0.516,0.675400
4,U005,D001,0.023722,0.151766,0.5,1.000000,0.585714,0.516,0.640257


In [42]:
# Cell 11 — Calculate hybrid score

all_hybrid["hybrid_score"] = (
    0.60 * all_hybrid["content_similarity_normalized"]
    + 0.40 * all_hybrid["context_score"]
)

print("Hybrid score calculated.")

print(
    "Hybrid score range:",
    all_hybrid["hybrid_score"].min(),
    "to",
    all_hybrid["hybrid_score"].max()
)

Hybrid score calculated.
Hybrid score range: 0.08418714285714285 to 0.98074


In [43]:
# Cell 13 — Rank recommendations for each user

all_hybrid["rank"] = (
    all_hybrid
    .groupby("user_id")["hybrid_score"]
    .rank(
        method="first",
        ascending=False
    )
)

print("Ranking completed.")

display(
    all_hybrid
    .sort_values(
        ["user_id", "rank"]
    )
    .head(20)
)

Ranking completed.


,user_id,destination_id,content_similarity,content_similarity_normalized,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score,hybrid_score,rank
2500,U001,D051,0.094431,0.970480,1.0,1.000000,0.585714,0.803,0.908307,0.945611,1.0
1550,U001,D032,0.087130,0.895445,1.0,1.000000,0.528571,0.896,0.913686,0.902741,2.0
3450,U001,D070,0.076947,0.790800,1.0,1.000000,0.471429,0.944,0.912314,0.839406,3.0
1650,U001,D034,0.080386,0.826144,1.0,1.000000,0.471429,0.520,0.848714,0.835172,4.0
4800,U001,D097,0.076016,0.781224,1.0,1.000000,0.600000,0.836,0.915400,0.834895,5.0
4600,U001,D093,0.080273,0.824979,1.0,0.833333,0.514286,0.701,0.840626,0.831238,6.0
1300,U001,D027,0.079604,0.818102,1.0,1.000000,0.571429,0.393,0.844664,0.828727,7.0
3250,U001,D066,0.080032,0.822499,1.0,1.000000,0.514286,0.389,0.835493,0.827697,8.0
1150,U001,D024,0.069081,0.709952,1.0,1.000000,0.557143,0.946,0.925471,0.796159,9.0
1000,U001,D021,0.073411,0.754452,1.0,1.000000,0.442857,0.583,0.853879,0.794223,10.0


In [44]:
# Cell 12 — Check hybrid dataframe

print("Shape:", all_hybrid.shape)

print("\nColumns:")
print(all_hybrid.columns.tolist())

print("\nUsers:", all_hybrid["user_id"].nunique())

print(
    "Destinations:",
    all_hybrid["destination_id"].nunique()
)

display(all_hybrid.head(10))

Shape: (5000, 11)

Columns:
['user_id', 'destination_id', 'content_similarity', 'content_similarity_normalized', 'preference_score', 'budget_score', 'crowd_fit_score', 'weather_fit_score', 'context_score', 'hybrid_score', 'rank']

Users: 50
Destinations: 100


,user_id,destination_id,content_similarity,content_similarity_normalized,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score,hybrid_score,rank
0,U001,D001,0.021762,0.223653,0.5,0.416667,0.414286,0.516,0.468710,0.321675,79.0
1,U002,D001,0.000000,0.000000,0.0,0.666667,0.820000,0.516,0.367067,0.146827,92.0
2,U003,D001,0.124593,0.760817,1.0,0.916667,0.414286,0.516,0.818710,0.783974,11.0
3,U004,D001,0.009276,0.072408,0.5,1.000000,0.820000,0.516,0.675400,0.313605,71.0
4,U005,D001,0.023722,0.151766,0.5,1.000000,0.585714,0.516,0.640257,0.347162,33.0
5,U006,D001,0.000000,0.000000,0.0,1.000000,0.414286,0.516,0.389543,0.155817,94.0
6,U007,D001,0.036466,0.287441,0.5,0.416667,0.820000,0.516,0.529567,0.384291,41.0
7,U008,D001,0.000000,0.000000,0.0,0.666667,0.414286,0.516,0.306210,0.122484,100.0
8,U009,D001,0.048664,0.237840,0.5,0.916667,0.820000,0.516,0.654567,0.404531,27.0
9,U010,D001,0.011298,0.065048,0.0,1.000000,0.585714,0.516,0.415257,0.205131,67.0


In [45]:
# Cell 14 — Select top 10 destinations per user

b4_top10 = (
    all_hybrid[
        all_hybrid["rank"] <= 10
    ]
    .sort_values(
        ["user_id", "rank"]
    )
    .copy()
)

print("B4 Top 10 shape:", b4_top10.shape)

print(
    "Users:",
    b4_top10["user_id"].nunique()
)

display(b4_top10.head(10))

B4 Top 10 shape: (500, 11)
Users: 50


,user_id,destination_id,content_similarity,content_similarity_normalized,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score,hybrid_score,rank
2500,U001,D051,0.094431,0.970480,1.0,1.000000,0.585714,0.803,0.908307,0.945611,1.0
1550,U001,D032,0.087130,0.895445,1.0,1.000000,0.528571,0.896,0.913686,0.902741,2.0
3450,U001,D070,0.076947,0.790800,1.0,1.000000,0.471429,0.944,0.912314,0.839406,3.0
1650,U001,D034,0.080386,0.826144,1.0,1.000000,0.471429,0.520,0.848714,0.835172,4.0
4800,U001,D097,0.076016,0.781224,1.0,1.000000,0.600000,0.836,0.915400,0.834895,5.0
4600,U001,D093,0.080273,0.824979,1.0,0.833333,0.514286,0.701,0.840626,0.831238,6.0
1300,U001,D027,0.079604,0.818102,1.0,1.000000,0.571429,0.393,0.844664,0.828727,7.0
3250,U001,D066,0.080032,0.822499,1.0,1.000000,0.514286,0.389,0.835493,0.827697,8.0
1150,U001,D024,0.069081,0.709952,1.0,1.000000,0.557143,0.946,0.925471,0.796159,9.0
1000,U001,D021,0.073411,0.754452,1.0,1.000000,0.442857,0.583,0.853879,0.794223,10.0


In [46]:
# Cell 15 — Add destination information

destination_details = destinations[
    [
        "destination_id",
        "name",
        "category",
        "district",
        "estimated_cost_lkr",
        "duration_hours",
        "sustainability_score",
        "crowd_score",
        "seasonality_score",
        "weather_temp_c",
        "weather_rain_risk",
        "weather_suitability"
    ]
].copy()

b4_top10 = b4_top10.merge(
    destination_details,
    on="destination_id",
    how="left"
)

print("Destination details added.")

print("Shape:", b4_top10.shape)

display(b4_top10.head(10))

Destination details added.
Shape: (500, 22)


,user_id,destination_id,content_similarity,content_similarity_normalized,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score,hybrid_score,...,category,district,estimated_cost_lkr,duration_hours,sustainability_score,crowd_score,seasonality_score,weather_temp_c,weather_rain_risk,weather_suitability
0,U001,D051,0.094431,0.970480,1.0,1.000000,0.585714,0.803,0.908307,0.945611,...,nature;city,colombo,2000,2.0,58,29,56,28.4,0.197,0.803
1,U001,D032,0.087130,0.895445,1.0,1.000000,0.528571,0.896,0.913686,0.902741,...,nature,badulla,3000,2.0,65,33,59,30.1,0.104,0.896
2,U001,D070,0.076947,0.790800,1.0,1.000000,0.471429,0.944,0.912314,0.839406,...,nature,kalutara,4000,2.5,59,37,67,26.8,0.056,0.944
3,U001,D034,0.080386,0.826144,1.0,1.000000,0.471429,0.520,0.848714,0.835172,...,nature,badulla,4000,4.0,67,37,64,30.8,0.480,0.520
4,U001,D097,0.076016,0.781224,1.0,1.000000,0.600000,0.836,0.915400,0.834895,...,nature;city,badulla,4000,3.0,61,28,52,25.1,0.164,0.836
5,U001,D093,0.080273,0.824979,1.0,0.833333,0.514286,0.701,0.840626,0.831238,...,nature;adventure,matale,6000,5.0,57,34,58,24.8,0.299,0.701
6,U001,D027,0.079604,0.818102,1.0,1.000000,0.571429,0.393,0.844664,0.828727,...,nature,nuwara eliya,3000,2.0,65,30,58,26.9,0.607,0.393
7,U001,D066,0.080032,0.822499,1.0,1.000000,0.514286,0.389,0.835493,0.827697,...,nature,ratnapura,3000,3.0,57,34,56,28.7,0.611,0.389
8,U001,D024,0.069081,0.709952,1.0,1.000000,0.557143,0.946,0.925471,0.796159,...,nature,nuwara eliya,3500,3.0,62,31,57,25.1,0.054,0.946
9,U001,D021,0.073411,0.754452,1.0,1.000000,0.442857,0.583,0.853879,0.794223,...,nature,nuwara eliya,3000,2.5,61,39,53,24.2,0.417,0.583


In [47]:
# Cell 16 — Generate recommendation explanations

def create_explanation(row):

    reasons = []

    if row["preference_score"] > 0:
        reasons.append("matches your interests")

    if row["budget_score"] >= 0.8:
        reasons.append("fits your budget")

    if row["crowd_fit_score"] >= 0.6:
        reasons.append("matches your preferred crowd level")

    if row["weather_fit_score"] >= 0.7:
        reasons.append("has suitable weather")

    if not reasons:
        reasons.append("has a reasonable overall match")

    return "; ".join(reasons)


b4_top10["explanation"] = (
    b4_top10.apply(
        create_explanation,
        axis=1
    )
)

print("Explanations created.")

display(
    b4_top10[
        [
            "user_id",
            "destination_id",
            "name",
            "hybrid_score",
            "explanation"
        ]
    ].head(10)
)

Explanations created.


,user_id,destination_id,name,hybrid_score,explanation
0,U001,D051,Viharamahadevi Park,0.945611,matches your interests; fits your budget; has ...
1,U001,D032,Ravana Falls,0.902741,matches your interests; fits your budget; has ...
2,U001,D070,Brief Garden,0.839406,matches your interests; fits your budget; has ...
3,U001,D034,Lipton's Seat,0.835172,matches your interests; fits your budget
4,U001,D097,Haputale,0.834895,matches your interests; fits your budget; matc...
5,U001,D093,Riverston,0.831238,matches your interests; fits your budget; has ...
6,U001,D027,Ramboda Falls,0.828727,matches your interests; fits your budget
7,U001,D066,Bopath Ella,0.827697,matches your interests; fits your budget
8,U001,D024,Hakgala Botanical Garden,0.796159,matches your interests; fits your budget; has ...
9,U001,D021,Gregory Lake,0.794223,matches your interests; fits your budget


In [48]:
# Cell 17 — Arrange final B4 columns

b4_final = b4_top10[
    [
        "user_id",
        "destination_id",
        "name",
        "category",
        "district",
        "estimated_cost_lkr",
        "duration_hours",
        "sustainability_score",
        "crowd_score",
        "seasonality_score",
        "weather_temp_c",
        "weather_rain_risk",
        "weather_suitability",
        "content_similarity",
        "content_similarity_normalized",
        "preference_score",
        "budget_score",
        "crowd_fit_score",
        "weather_fit_score",
        "context_score",
        "hybrid_score",
        "rank",
        "explanation"
    ]
].copy()

print("Final B4 columns:")
print(b4_final.columns.tolist())

print("\nShape:")
print(b4_final.shape)

Final B4 columns:
['user_id', 'destination_id', 'name', 'category', 'district', 'estimated_cost_lkr', 'duration_hours', 'sustainability_score', 'crowd_score', 'seasonality_score', 'weather_temp_c', 'weather_rain_risk', 'weather_suitability', 'content_similarity', 'content_similarity_normalized', 'preference_score', 'budget_score', 'crowd_fit_score', 'weather_fit_score', 'context_score', 'hybrid_score', 'rank', 'explanation']

Shape:
(500, 23)


In [49]:
# Cell 18 — Verify recommendation count

recommendation_counts = (
    b4_final
    .groupby("user_id")
    .size()
)

print("Number of users:", len(recommendation_counts))

print("\nRecommendations per user:")
print(
    recommendation_counts
    .value_counts()
    .sort_index()
)

print(
    "\nMinimum recommendations:",
    recommendation_counts.min()
)

print(
    "Maximum recommendations:",
    recommendation_counts.max()
)

Number of users: 50

Recommendations per user:
10    50
Name: count, dtype: int64

Minimum recommendations: 10
Maximum recommendations: 10


In [50]:
# Cell 19 — Score validation

print("Hybrid score range:")

print(
    "Minimum:",
    b4_final["hybrid_score"].min()
)

print(
    "Maximum:",
    b4_final["hybrid_score"].max()
)

print(
    "Mean:",
    b4_final["hybrid_score"].mean()
)

print("\nMissing values:")

print(
    b4_final.isna().sum()
)

Hybrid score range:
Minimum: 0.46399198451102
Maximum: 0.98074
Mean: 0.7901326672046292

Missing values:
user_id                          0
destination_id                   0
name                             0
category                         0
district                         0
estimated_cost_lkr               0
duration_hours                   0
sustainability_score             0
crowd_score                      0
seasonality_score                0
weather_temp_c                   0
weather_rain_risk                0
weather_suitability              0
content_similarity               0
content_similarity_normalized    0
preference_score                 0
budget_score                     0
crowd_fit_score                  0
weather_fit_score                0
context_score                    0
hybrid_score                     0
rank                             0
explanation                      0
dtype: int64


In [51]:
# Cell 20 — Example recommendations

test_user_id = users["user_id"].iloc[0]

print("Test user:", test_user_id)

display(
    b4_final[
        b4_final["user_id"] == test_user_id
    ][
        [
            "rank",
            "destination_id",
            "name",
            "category",
            "district",
            "hybrid_score",
            "explanation"
        ]
    ].sort_values("rank")
)

Test user: U001


,rank,destination_id,name,category,district,hybrid_score,explanation
0,1.0,D051,Viharamahadevi Park,nature;city,colombo,0.945611,matches your interests; fits your budget; has ...
1,2.0,D032,Ravana Falls,nature,badulla,0.902741,matches your interests; fits your budget; has ...
2,3.0,D070,Brief Garden,nature,kalutara,0.839406,matches your interests; fits your budget; has ...
3,4.0,D034,Lipton's Seat,nature,badulla,0.835172,matches your interests; fits your budget
4,5.0,D097,Haputale,nature;city,badulla,0.834895,matches your interests; fits your budget; matc...
5,6.0,D093,Riverston,nature;adventure,matale,0.831238,matches your interests; fits your budget; has ...
6,7.0,D027,Ramboda Falls,nature,nuwara eliya,0.828727,matches your interests; fits your budget
7,8.0,D066,Bopath Ella,nature,ratnapura,0.827697,matches your interests; fits your budget
8,9.0,D024,Hakgala Botanical Garden,nature,nuwara eliya,0.796159,matches your interests; fits your budget; has ...
9,10.0,D021,Gregory Lake,nature,nuwara eliya,0.794223,matches your interests; fits your budget


In [52]:
# Cell 21 — Create results folder

from pathlib import Path

RESULTS_DIR = Path("../results")

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Results folder:", RESULTS_DIR.resolve())

Results folder: C:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results


In [53]:
# Cell 22 — Save corrected B4

b4_file = (
    RESULTS_DIR /
    "b4_hybrid_recommendations.csv"
)

b4_final.to_csv(
    b4_file,
    index=False
)

print("B4 saved successfully:")
print(b4_file.resolve())

B4 saved successfully:
C:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results\b4_hybrid_recommendations.csv


In [54]:
# Cell 23 — Final verification

check_b4 = pd.read_csv(
    b4_file
)

print("========== FINAL B4 CHECK ==========")

print(
    "Rows:",
    len(check_b4)
)

print(
    "Users:",
    check_b4["user_id"].nunique()
)

print(
    "Destinations:",
    check_b4["destination_id"].nunique()
)

print(
    "Min rank:",
    check_b4["rank"].min()
)

print(
    "Max rank:",
    check_b4["rank"].max()
)

print("\nRows per user:")
print(
    check_b4
    .groupby("user_id")
    .size()
    .value_counts()
    .sort_index()
)

========== FINAL B4 CHECK ==========
Rows: 500
Users: 50
Destinations: 76
Min rank: 1.0
Max rank: 10.0

Rows per user:
10    50
Name: count, dtype: int64


In [55]:
all_hybrid["hybrid_score"] = (
    0.60 * all_hybrid["content_similarity_normalized"]
    + 0.40 * all_hybrid["context_score"]
)

print("Hybrid score calculated.")

print(
    "Hybrid score range:",
    all_hybrid["hybrid_score"].min(),
    "to",
    all_hybrid["hybrid_score"].max()
)

Hybrid score calculated.
Hybrid score range: 0.08418714285714285 to 0.98074


In [56]:
all_hybrid["hybrid_score"] = (
    0.60 * all_hybrid["content_similarity_normalized"]
    + 0.40 * all_hybrid["context_score"]
)

print("Hybrid score calculated.")

print(
    "Hybrid score range:",
    all_hybrid["hybrid_score"].min(),
    "to",
    all_hybrid["hybrid_score"].max()
)

Hybrid score calculated.
Hybrid score range: 0.08418714285714285 to 0.98074


In [57]:
print("========== B4 DATA CHECK ==========")

print("Shape:", all_hybrid.shape)

print("\nColumns:")
print(all_hybrid.columns.tolist())

print("\nNumber of users:")
print(all_hybrid["user_id"].nunique())

print("\nNumber of destinations:")
print(all_hybrid["destination_id"].nunique())

print("\nNumber of user-destination pairs:")
print(len(all_hybrid))

print("\nMissing values:")
print(all_hybrid.isna().sum())

display(all_hybrid.head(10))

========== B4 DATA CHECK ==========
Shape: (5000, 11)

Columns:
['user_id', 'destination_id', 'content_similarity', 'content_similarity_normalized', 'preference_score', 'budget_score', 'crowd_fit_score', 'weather_fit_score', 'context_score', 'hybrid_score', 'rank']

Number of users:
50

Number of destinations:
100

Number of user-destination pairs:
5000

Missing values:
user_id                          0
destination_id                   0
content_similarity               0
content_similarity_normalized    0
preference_score                 0
budget_score                     0
crowd_fit_score                  0
weather_fit_score                0
context_score                    0
hybrid_score                     0
rank                             0
dtype: int64


,user_id,destination_id,content_similarity,content_similarity_normalized,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score,hybrid_score,rank
0,U001,D001,0.021762,0.223653,0.5,0.416667,0.414286,0.516,0.468710,0.321675,79.0
1,U002,D001,0.000000,0.000000,0.0,0.666667,0.820000,0.516,0.367067,0.146827,92.0
2,U003,D001,0.124593,0.760817,1.0,0.916667,0.414286,0.516,0.818710,0.783974,11.0
3,U004,D001,0.009276,0.072408,0.5,1.000000,0.820000,0.516,0.675400,0.313605,71.0
4,U005,D001,0.023722,0.151766,0.5,1.000000,0.585714,0.516,0.640257,0.347162,33.0
5,U006,D001,0.000000,0.000000,0.0,1.000000,0.414286,0.516,0.389543,0.155817,94.0
6,U007,D001,0.036466,0.287441,0.5,0.416667,0.820000,0.516,0.529567,0.384291,41.0
7,U008,D001,0.000000,0.000000,0.0,0.666667,0.414286,0.516,0.306210,0.122484,100.0
8,U009,D001,0.048664,0.237840,0.5,0.916667,0.820000,0.516,0.654567,0.404531,27.0
9,U010,D001,0.011298,0.065048,0.0,1.000000,0.585714,0.516,0.415257,0.205131,67.0


In [58]:
duplicate_pairs = all_hybrid.duplicated(
    subset=["user_id", "destination_id"]
).sum()

print("Duplicate user-destination pairs:", duplicate_pairs)

Duplicate user-destination pairs: 0


In [59]:
print("========== SCORE CHECK ==========")

print("\nContent similarity:")
print(
    all_hybrid["content_similarity"].min(),
    "to",
    all_hybrid["content_similarity"].max()
)

print("\nNormalized content similarity:")
print(
    all_hybrid["content_similarity_normalized"].min(),
    "to",
    all_hybrid["content_similarity_normalized"].max()
)

print("\nContext score:")
print(
    all_hybrid["context_score"].min(),
    "to",
    all_hybrid["context_score"].max()
)

print("\nHybrid score:")
print(
    all_hybrid["hybrid_score"].min(),
    "to",
    all_hybrid["hybrid_score"].max()
)

========== SCORE CHECK ==========

Content similarity:
0.0 to 0.22740392549007252

Normalized content similarity:
0.0 to 1.0

Context score:
0.21046785714285712 to 0.98395

Hybrid score:
0.08418714285714285 to 0.98074


In [60]:
all_hybrid["rank"] = (
    all_hybrid
    .groupby("user_id")["hybrid_score"]
    .rank(
        method="first",
        ascending=False
    )
)

print("Ranking completed.")

display(
    all_hybrid
    .sort_values(["user_id", "rank"])
    .head(20)
)

Ranking completed.


,user_id,destination_id,content_similarity,content_similarity_normalized,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score,hybrid_score,rank
2500,U001,D051,0.094431,0.970480,1.0,1.000000,0.585714,0.803,0.908307,0.945611,1.0
1550,U001,D032,0.087130,0.895445,1.0,1.000000,0.528571,0.896,0.913686,0.902741,2.0
3450,U001,D070,0.076947,0.790800,1.0,1.000000,0.471429,0.944,0.912314,0.839406,3.0
1650,U001,D034,0.080386,0.826144,1.0,1.000000,0.471429,0.520,0.848714,0.835172,4.0
4800,U001,D097,0.076016,0.781224,1.0,1.000000,0.600000,0.836,0.915400,0.834895,5.0
4600,U001,D093,0.080273,0.824979,1.0,0.833333,0.514286,0.701,0.840626,0.831238,6.0
1300,U001,D027,0.079604,0.818102,1.0,1.000000,0.571429,0.393,0.844664,0.828727,7.0
3250,U001,D066,0.080032,0.822499,1.0,1.000000,0.514286,0.389,0.835493,0.827697,8.0
1150,U001,D024,0.069081,0.709952,1.0,1.000000,0.557143,0.946,0.925471,0.796159,9.0
1000,U001,D021,0.073411,0.754452,1.0,1.000000,0.442857,0.583,0.853879,0.794223,10.0


In [61]:
rank_check = (
    all_hybrid
    .groupby("user_id")["rank"]
    .agg(["min", "max", "count"])
)

print(rank_check)

         min    max  count
user_id                   
U001     1.0  100.0    100
U002     1.0  100.0    100
U003     1.0  100.0    100
U004     1.0  100.0    100
U005     1.0  100.0    100
U006     1.0  100.0    100
U007     1.0  100.0    100
U008     1.0  100.0    100
U009     1.0  100.0    100
U010     1.0  100.0    100
U011     1.0  100.0    100
U012     1.0  100.0    100
U013     1.0  100.0    100
U014     1.0  100.0    100
U015     1.0  100.0    100
U016     1.0  100.0    100
U017     1.0  100.0    100
U018     1.0  100.0    100
U019     1.0  100.0    100
U020     1.0  100.0    100
U021     1.0  100.0    100
U022     1.0  100.0    100
U023     1.0  100.0    100
U024     1.0  100.0    100
U025     1.0  100.0    100
U026     1.0  100.0    100
U027     1.0  100.0    100
U028     1.0  100.0    100
U029     1.0  100.0    100
U030     1.0  100.0    100
U031     1.0  100.0    100
U032     1.0  100.0    100
U033     1.0  100.0    100
U034     1.0  100.0    100
U035     1.0  100.0    100
U

In [62]:
b4_top10 = (
    all_hybrid[
        all_hybrid["rank"] <= 10
    ]
    .sort_values(["user_id", "rank"])
    .copy()
)

print("B4 Top 10 shape:", b4_top10.shape)

print("Number of users:", b4_top10["user_id"].nunique())

print("Minimum rank:", b4_top10["rank"].min())

print("Maximum rank:", b4_top10["rank"].max())

display(b4_top10.head(10))

B4 Top 10 shape: (500, 11)
Number of users: 50
Minimum rank: 1.0
Maximum rank: 10.0


,user_id,destination_id,content_similarity,content_similarity_normalized,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score,hybrid_score,rank
2500,U001,D051,0.094431,0.970480,1.0,1.000000,0.585714,0.803,0.908307,0.945611,1.0
1550,U001,D032,0.087130,0.895445,1.0,1.000000,0.528571,0.896,0.913686,0.902741,2.0
3450,U001,D070,0.076947,0.790800,1.0,1.000000,0.471429,0.944,0.912314,0.839406,3.0
1650,U001,D034,0.080386,0.826144,1.0,1.000000,0.471429,0.520,0.848714,0.835172,4.0
4800,U001,D097,0.076016,0.781224,1.0,1.000000,0.600000,0.836,0.915400,0.834895,5.0
4600,U001,D093,0.080273,0.824979,1.0,0.833333,0.514286,0.701,0.840626,0.831238,6.0
1300,U001,D027,0.079604,0.818102,1.0,1.000000,0.571429,0.393,0.844664,0.828727,7.0
3250,U001,D066,0.080032,0.822499,1.0,1.000000,0.514286,0.389,0.835493,0.827697,8.0
1150,U001,D024,0.069081,0.709952,1.0,1.000000,0.557143,0.946,0.925471,0.796159,9.0
1000,U001,D021,0.073411,0.754452,1.0,1.000000,0.442857,0.583,0.853879,0.794223,10.0


In [63]:
recommendation_counts = (
    b4_top10
    .groupby("user_id")
    .size()
)

print("Number of users:", len(recommendation_counts))

print("\nRecommendations per user:")
print(
    recommendation_counts
    .value_counts()
    .sort_index()
)

print("\nMinimum recommendations:", recommendation_counts.min())

print("Maximum recommendations:", recommendation_counts.max())

Number of users: 50

Recommendations per user:
10    50
Name: count, dtype: int64

Minimum recommendations: 10
Maximum recommendations: 10


In [64]:
destination_details = destinations[
    [
        "destination_id",
        "name",
        "category",
        "district",
        "estimated_cost_lkr",
        "duration_hours",
        "sustainability_score",
        "crowd_score",
        "seasonality_score",
        "weather_temp_c",
        "weather_rain_risk",
        "weather_suitability"
    ]
].copy()

b4_top10 = b4_top10.merge(
    destination_details,
    on="destination_id",
    how="left"
)

print("Destination details added.")

print("Shape:", b4_top10.shape)

display(b4_top10.head(10))

Destination details added.
Shape: (500, 22)


,user_id,destination_id,content_similarity,content_similarity_normalized,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score,hybrid_score,...,category,district,estimated_cost_lkr,duration_hours,sustainability_score,crowd_score,seasonality_score,weather_temp_c,weather_rain_risk,weather_suitability
0,U001,D051,0.094431,0.970480,1.0,1.000000,0.585714,0.803,0.908307,0.945611,...,nature;city,colombo,2000,2.0,58,29,56,28.4,0.197,0.803
1,U001,D032,0.087130,0.895445,1.0,1.000000,0.528571,0.896,0.913686,0.902741,...,nature,badulla,3000,2.0,65,33,59,30.1,0.104,0.896
2,U001,D070,0.076947,0.790800,1.0,1.000000,0.471429,0.944,0.912314,0.839406,...,nature,kalutara,4000,2.5,59,37,67,26.8,0.056,0.944
3,U001,D034,0.080386,0.826144,1.0,1.000000,0.471429,0.520,0.848714,0.835172,...,nature,badulla,4000,4.0,67,37,64,30.8,0.480,0.520
4,U001,D097,0.076016,0.781224,1.0,1.000000,0.600000,0.836,0.915400,0.834895,...,nature;city,badulla,4000,3.0,61,28,52,25.1,0.164,0.836
5,U001,D093,0.080273,0.824979,1.0,0.833333,0.514286,0.701,0.840626,0.831238,...,nature;adventure,matale,6000,5.0,57,34,58,24.8,0.299,0.701
6,U001,D027,0.079604,0.818102,1.0,1.000000,0.571429,0.393,0.844664,0.828727,...,nature,nuwara eliya,3000,2.0,65,30,58,26.9,0.607,0.393
7,U001,D066,0.080032,0.822499,1.0,1.000000,0.514286,0.389,0.835493,0.827697,...,nature,ratnapura,3000,3.0,57,34,56,28.7,0.611,0.389
8,U001,D024,0.069081,0.709952,1.0,1.000000,0.557143,0.946,0.925471,0.796159,...,nature,nuwara eliya,3500,3.0,62,31,57,25.1,0.054,0.946
9,U001,D021,0.073411,0.754452,1.0,1.000000,0.442857,0.583,0.853879,0.794223,...,nature,nuwara eliya,3000,2.5,61,39,53,24.2,0.417,0.583


In [65]:
print("Missing destination names:",
      b4_top10["name"].isna().sum())

print("Missing categories:",
      b4_top10["category"].isna().sum())

print("Missing districts:",
      b4_top10["district"].isna().sum())

print("\nUnique destinations in B4:",
      b4_top10["destination_id"].nunique())

Missing destination names: 0
Missing categories: 0
Missing districts: 0

Unique destinations in B4: 76


In [66]:
def create_explanation(row):
    reasons = []

    if row["preference_score"] > 0:
        reasons.append("matches your interests")

    if row["budget_score"] >= 0.8:
        reasons.append("fits your budget")

    if row["crowd_fit_score"] >= 0.6:
        reasons.append("matches your preferred crowd level")

    if row["weather_fit_score"] >= 0.7:
        reasons.append("has suitable weather")

    if not reasons:
        reasons.append("has a reasonable overall match")

    return "; ".join(reasons)


b4_top10["explanation"] = b4_top10.apply(
    create_explanation,
    axis=1
)

print("Explanations created.")

display(
    b4_top10[
        [
            "user_id",
            "destination_id",
            "name",
            "hybrid_score",
            "explanation"
        ]
    ].head(10)
)

Explanations created.


,user_id,destination_id,name,hybrid_score,explanation
0,U001,D051,Viharamahadevi Park,0.945611,matches your interests; fits your budget; has ...
1,U001,D032,Ravana Falls,0.902741,matches your interests; fits your budget; has ...
2,U001,D070,Brief Garden,0.839406,matches your interests; fits your budget; has ...
3,U001,D034,Lipton's Seat,0.835172,matches your interests; fits your budget
4,U001,D097,Haputale,0.834895,matches your interests; fits your budget; matc...
5,U001,D093,Riverston,0.831238,matches your interests; fits your budget; has ...
6,U001,D027,Ramboda Falls,0.828727,matches your interests; fits your budget
7,U001,D066,Bopath Ella,0.827697,matches your interests; fits your budget
8,U001,D024,Hakgala Botanical Garden,0.796159,matches your interests; fits your budget; has ...
9,U001,D021,Gregory Lake,0.794223,matches your interests; fits your budget


In [67]:
b4_final = b4_top10[
    [
        "user_id",
        "destination_id",
        "name",
        "category",
        "district",
        "estimated_cost_lkr",
        "duration_hours",
        "sustainability_score",
        "crowd_score",
        "seasonality_score",
        "weather_temp_c",
        "weather_rain_risk",
        "weather_suitability",
        "content_similarity",
        "content_similarity_normalized",
        "preference_score",
        "budget_score",
        "crowd_fit_score",
        "weather_fit_score",
        "context_score",
        "hybrid_score",
        "rank",
        "explanation"
    ]
].copy()

print("========== FINAL B4 ==========")

print("Columns:")
print(b4_final.columns.tolist())

print("\nShape:")
print(b4_final.shape)

display(b4_final.head(10))

========== FINAL B4 ==========
Columns:
['user_id', 'destination_id', 'name', 'category', 'district', 'estimated_cost_lkr', 'duration_hours', 'sustainability_score', 'crowd_score', 'seasonality_score', 'weather_temp_c', 'weather_rain_risk', 'weather_suitability', 'content_similarity', 'content_similarity_normalized', 'preference_score', 'budget_score', 'crowd_fit_score', 'weather_fit_score', 'context_score', 'hybrid_score', 'rank', 'explanation']

Shape:
(500, 23)


,user_id,destination_id,name,category,district,estimated_cost_lkr,duration_hours,sustainability_score,crowd_score,seasonality_score,...,content_similarity,content_similarity_normalized,preference_score,budget_score,crowd_fit_score,weather_fit_score,context_score,hybrid_score,rank,explanation
0,U001,D051,Viharamahadevi Park,nature;city,colombo,2000,2.0,58,29,56,...,0.094431,0.970480,1.0,1.000000,0.585714,0.803,0.908307,0.945611,1.0,matches your interests; fits your budget; has ...
1,U001,D032,Ravana Falls,nature,badulla,3000,2.0,65,33,59,...,0.087130,0.895445,1.0,1.000000,0.528571,0.896,0.913686,0.902741,2.0,matches your interests; fits your budget; has ...
2,U001,D070,Brief Garden,nature,kalutara,4000,2.5,59,37,67,...,0.076947,0.790800,1.0,1.000000,0.471429,0.944,0.912314,0.839406,3.0,matches your interests; fits your budget; has ...
3,U001,D034,Lipton's Seat,nature,badulla,4000,4.0,67,37,64,...,0.080386,0.826144,1.0,1.000000,0.471429,0.520,0.848714,0.835172,4.0,matches your interests; fits your budget
4,U001,D097,Haputale,nature;city,badulla,4000,3.0,61,28,52,...,0.076016,0.781224,1.0,1.000000,0.600000,0.836,0.915400,0.834895,5.0,matches your interests; fits your budget; matc...
5,U001,D093,Riverston,nature;adventure,matale,6000,5.0,57,34,58,...,0.080273,0.824979,1.0,0.833333,0.514286,0.701,0.840626,0.831238,6.0,matches your interests; fits your budget; has ...
6,U001,D027,Ramboda Falls,nature,nuwara eliya,3000,2.0,65,30,58,...,0.079604,0.818102,1.0,1.000000,0.571429,0.393,0.844664,0.828727,7.0,matches your interests; fits your budget
7,U001,D066,Bopath Ella,nature,ratnapura,3000,3.0,57,34,56,...,0.080032,0.822499,1.0,1.000000,0.514286,0.389,0.835493,0.827697,8.0,matches your interests; fits your budget
8,U001,D024,Hakgala Botanical Garden,nature,nuwara eliya,3500,3.0,62,31,57,...,0.069081,0.709952,1.0,1.000000,0.557143,0.946,0.925471,0.796159,9.0,matches your interests; fits your budget; has ...
9,U001,D021,Gregory Lake,nature,nuwara eliya,3000,2.5,61,39,53,...,0.073411,0.754452,1.0,1.000000,0.442857,0.583,0.853879,0.794223,10.0,matches your interests; fits your budget


In [68]:
print("========== FINAL B4 QUALITY CHECK ==========")

print("Rows:", len(b4_final))

print("Users:", b4_final["user_id"].nunique())

print("Destinations:", b4_final["destination_id"].nunique())

print("Minimum rank:", b4_final["rank"].min())

print("Maximum rank:", b4_final["rank"].max())

print("\nRows per user:")
print(
    b4_final
    .groupby("user_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nMissing values:")
print(
    b4_final.isna().sum()
)

========== FINAL B4 QUALITY CHECK ==========
Rows: 500
Users: 50
Destinations: 76
Minimum rank: 1.0
Maximum rank: 10.0

Rows per user:
10    50
Name: count, dtype: int64

Missing values:
user_id                          0
destination_id                   0
name                             0
category                         0
district                         0
estimated_cost_lkr               0
duration_hours                   0
sustainability_score             0
crowd_score                      0
seasonality_score                0
weather_temp_c                   0
weather_rain_risk                0
weather_suitability              0
content_similarity               0
content_similarity_normalized    0
preference_score                 0
budget_score                     0
crowd_fit_score                  0
weather_fit_score                0
context_score                    0
hybrid_score                     0
rank                             0
explanation                      0
dtype: i

In [69]:
test_user_id = users["user_id"].iloc[0]

print("Test user:", test_user_id)

display(
    b4_final[
        b4_final["user_id"] == test_user_id
    ][
        [
            "rank",
            "destination_id",
            "name",
            "category",
            "district",
            "hybrid_score",
            "explanation"
        ]
    ]
    .sort_values("rank")
)

Test user: U001


,rank,destination_id,name,category,district,hybrid_score,explanation
0,1.0,D051,Viharamahadevi Park,nature;city,colombo,0.945611,matches your interests; fits your budget; has ...
1,2.0,D032,Ravana Falls,nature,badulla,0.902741,matches your interests; fits your budget; has ...
2,3.0,D070,Brief Garden,nature,kalutara,0.839406,matches your interests; fits your budget; has ...
3,4.0,D034,Lipton's Seat,nature,badulla,0.835172,matches your interests; fits your budget
4,5.0,D097,Haputale,nature;city,badulla,0.834895,matches your interests; fits your budget; matc...
5,6.0,D093,Riverston,nature;adventure,matale,0.831238,matches your interests; fits your budget; has ...
6,7.0,D027,Ramboda Falls,nature,nuwara eliya,0.828727,matches your interests; fits your budget
7,8.0,D066,Bopath Ella,nature,ratnapura,0.827697,matches your interests; fits your budget
8,9.0,D024,Hakgala Botanical Garden,nature,nuwara eliya,0.796159,matches your interests; fits your budget; has ...
9,10.0,D021,Gregory Lake,nature,nuwara eliya,0.794223,matches your interests; fits your budget


In [70]:
test_scores = (
    b4_final[
        b4_final["user_id"] == test_user_id
    ]
    .sort_values("rank")["hybrid_score"]
    .values
)

print("Scores:")
print(test_scores)

print(
    "\nScores correctly descending:",
    all(
        test_scores[i] >= test_scores[i + 1]
        for i in range(len(test_scores) - 1)
    )
)

Scores:
[0.9456111  0.9027413  0.8394058  0.83517208 0.83489457 0.83123816
 0.82872692 0.82769673 0.79615949 0.79422276]

Scores correctly descending: True


In [71]:
from pathlib import Path

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Results folder:")
print(RESULTS_DIR.resolve())

Results folder:
C:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results


In [72]:
b4_file = RESULTS_DIR / "b4_hybrid_recommendations.csv"

b4_final.to_csv(
    b4_file,
    index=False
)

print("B4 saved successfully:")
print(b4_file.resolve())

B4 saved successfully:
C:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results\b4_hybrid_recommendations.csv


In [73]:
check_b4 = pd.read_csv(b4_file)

print("========== SAVED B4 CHECK ==========")

print("Rows:", len(check_b4))

print("Users:", check_b4["user_id"].nunique())

print("Destinations:", check_b4["destination_id"].nunique())

print("Minimum rank:", check_b4["rank"].min())

print("Maximum rank:", check_b4["rank"].max())

print("\nRows per user:")

print(
    check_b4
    .groupby("user_id")
    .size()
    .value_counts()
    .sort_index()
)

========== SAVED B4 CHECK ==========
Rows: 500
Users: 50
Destinations: 76
Minimum rank: 1.0
Maximum rank: 10.0

Rows per user:
10    50
Name: count, dtype: int64


In [74]:
b4_file = RESULTS_DIR / "b4_hybrid_recommendations.csv"

b4 = pd.read_csv(b4_file)

print("B4 loaded successfully")
print("Rows:", len(b4))
print("Users:", b4["user_id"].nunique())
print("Destinations:", b4["destination_id"].nunique())
print("Columns:")
print(b4.columns.tolist())

B4 loaded successfully
Rows: 500
Users: 50
Destinations: 76
Columns:
['user_id', 'destination_id', 'name', 'category', 'district', 'estimated_cost_lkr', 'duration_hours', 'sustainability_score', 'crowd_score', 'seasonality_score', 'weather_temp_c', 'weather_rain_risk', 'weather_suitability', 'content_similarity', 'content_similarity_normalized', 'preference_score', 'budget_score', 'crowd_fit_score', 'weather_fit_score', 'context_score', 'hybrid_score', 'rank', 'explanation']


In [75]:
b4_file = RESULTS_DIR / "b4_hybrid_recommendations.csv"

b4 = pd.read_csv(b4_file)

print("B4 loaded successfully")
print("Rows:", len(b4))
print("Users:", b4["user_id"].nunique())
print("Destinations:", b4["destination_id"].nunique())
print("Columns:")
print(b4.columns.tolist())

B4 loaded successfully
Rows: 500
Users: 50
Destinations: 76
Columns:
['user_id', 'destination_id', 'name', 'category', 'district', 'estimated_cost_lkr', 'duration_hours', 'sustainability_score', 'crowd_score', 'seasonality_score', 'weather_temp_c', 'weather_rain_risk', 'weather_suitability', 'content_similarity', 'content_similarity_normalized', 'preference_score', 'budget_score', 'crowd_fit_score', 'weather_fit_score', 'context_score', 'hybrid_score', 'rank', 'explanation']


In [76]:
b2_file = RESULTS_DIR / "b2_content_based_recommendations.csv"

b2 = pd.read_csv(b2_file)

print("B2 loaded successfully")
print("Shape:", b2.shape)
print("Columns:")
print(b2.columns.tolist())

display(b2.head())

B2 loaded successfully
Shape: (10, 5)
Columns:
['destination_id', 'name', 'category', 'district', 'content_similarity']


,destination_id,name,category,district,content_similarity
0,D032,Ravana Falls,nature,badulla,0.306305
1,D027,Ramboda Falls,nature,nuwara eliya,0.288402
2,D034,Lipton's Seat,nature,badulla,0.278799
3,D093,Riverston,nature;adventure,matale,0.276014
4,D066,Bopath Ella,nature,ratnapura,0.275734


In [78]:
print("B2 is already loaded.")
print("Shape:", b2.shape)
print("Columns:", b2.columns.tolist())
display(b2.head())

B2 is already loaded.
Shape: (10, 5)
Columns: ['destination_id', 'name', 'category', 'district', 'content_similarity']


,destination_id,name,category,district,content_similarity
0,D032,Ravana Falls,nature,badulla,0.306305
1,D027,Ramboda Falls,nature,nuwara eliya,0.288402
2,D034,Lipton's Seat,nature,badulla,0.278799
3,D093,Riverston,nature;adventure,matale,0.276014
4,D066,Bopath Ella,nature,ratnapura,0.275734


In [79]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("TfidfVectorizer imported successfully.")

TfidfVectorizer imported successfully.


In [80]:
users["tfidf_text"] = users["user_semantic_text"].fillna("")

print(
    "User TF-IDF text rows:",
    len(users)
)

display(
    users[
        ["user_id", "tfidf_text"]
    ].head(5)
)

KeyError: 'user_semantic_text'

In [81]:
print("Users shape:", users.shape)
print("\nUser columns:")
print(users.columns.tolist())

Users shape: (50, 13)

User columns:
['user_id', 'age_group', 'traveller_type', 'travel_style', 'budget_lkr_per_day', 'trip_duration_days', 'starting_location', 'preferred_interests', 'preferred_crowd', 'preferred_weather', 'sustainability_importance', 'data_status', 'num_preferred_interests']


In [82]:
# Cell 1 — Create user semantic text

def create_user_semantic_text(row):
    interests = str(row["preferred_interests"]).replace(";", ", ")

    return (
        f"Traveller type: {row['traveller_type']}. "
        f"Travel style: {row['travel_style']}. "
        f"Preferred interests: {interests}. "
        f"Preferred crowd: {row['preferred_crowd']}. "
        f"Preferred weather: {row['preferred_weather']}. "
        f"Starting location: {row['starting_location']}. "
        f"Budget: {row['budget_lkr_per_day']} LKR per day. "
        f"Trip duration: {row['trip_duration_days']} days. "
        f"Sustainability importance: {row['sustainability_importance']}."
    )

users["user_semantic_text"] = users.apply(
    create_user_semantic_text,
    axis=1
)

print("User semantic text created successfully.")
print("Shape:", users.shape)
print("\nExample:")
print(users[["user_id", "user_semantic_text"]].head(2).to_string(index=False))

User semantic text created successfully.
Shape: (50, 14)

Example:
user_id                                                                                                                                                                                                                                    user_semantic_text
   U001     Traveller type: solo. Travel style: budget. Preferred interests: nature, photography. Preferred crowd: low. Preferred weather: cool. Starting location: colombo. Budget: 5000 LKR per day. Trip duration: 1 days. Sustainability importance: 0.4.
   U002 Traveller type: couple. Travel style: moderate. Preferred interests: beach, relaxation. Preferred crowd: medium. Preferred weather: warm. Starting location: kandy. Budget: 8000 LKR per day. Trip duration: 2 days. Sustainability importance: 0.48.


In [83]:
# Cell 2 — Create destination semantic text

print("Destinations shape:", destinations.shape)
print("Destination columns:")
print(destinations.columns.tolist())

Destinations shape: (100, 21)
Destination columns:
['destination_id', 'name', 'district', 'latitude', 'longitude', 'category', 'estimated_cost_lkr', 'duration_hours', 'interest_tags', 'sustainability_score', 'crowd_score', 'seasonality_score', 'sentiment_nature', 'sentiment_service', 'sentiment_value', 'weather_temp_c', 'weather_rain_risk', 'weather_suitability', 'source_provenance', 'data_status', 'num_interest_tags']


In [84]:
# Create destination semantic text

def create_destination_semantic_text(row):
    interests = str(row["interest_tags"]).replace(";", ", ")

    return (
        f"Destination: {row['name']}. "
        f"Category: {row['category']}. "
        f"District: {row['district']}. "
        f"Suitable interests: {interests}. "
        f"Estimated cost: {row['estimated_cost_lkr']} LKR. "
        f"Duration: {row['duration_hours']} hours. "
        f"Sustainability score: {row['sustainability_score']}. "
        f"Crowd score: {row['crowd_score']}. "
        f"Seasonality score: {row['seasonality_score']}. "
        f"Temperature: {row['weather_temp_c']} C. "
        f"Rain risk: {row['weather_rain_risk']}. "
        f"Weather suitability: {row['weather_suitability']}."
    )

destinations["destination_semantic_text"] = destinations.apply(
    create_destination_semantic_text,
    axis=1
)

print("Destination semantic text created successfully.")
print("Shape:", destinations.shape)

Destination semantic text created successfully.
Shape: (100, 22)


In [85]:
# Cell 3 — Verify semantic text

print("Users:")
print("Shape:", users.shape)
print("Has user_semantic_text:", "user_semantic_text" in users.columns)

print("\nDestinations:")
print("Shape:", destinations.shape)
print(
    "Has destination_semantic_text:",
    "destination_semantic_text" in destinations.columns
)

print("\nUser text example:")
print(users.loc[0, "user_semantic_text"])

print("\nDestination text example:")
print(destinations.loc[0, "destination_semantic_text"])

Users:
Shape: (50, 14)
Has user_semantic_text: True

Destinations:
Shape: (100, 22)
Has destination_semantic_text: True

User text example:
Traveller type: solo. Travel style: budget. Preferred interests: nature, photography. Preferred crowd: low. Preferred weather: cool. Starting location: colombo. Budget: 5000 LKR per day. Trip duration: 1 days. Sustainability importance: 0.4.

Destination text example:
Destination: Sigiriya Rock Fortress. Category: heritage;nature. District: matale. Suitable interests: heritage, history, photography, hiking. Estimated cost: 12000 LKR. Duration: 3.0 hours. Sustainability score: 57. Crowd score: 41. Seasonality score: 58. Temperature: 25.1 C. Rain risk: 0.484. Weather suitability: 0.516.


In [86]:
# Cell 4 — Import SBERT

from sentence_transformers import SentenceTransformer

print("SentenceTransformer imported successfully.")

SentenceTransformer imported successfully.


In [87]:
# Cell 5 — Load SBERT model

model = SentenceTransformer("all-MiniLM-L6-v2")

print("SBERT model loaded successfully.")
print("Model:", model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SBERT model loaded successfully.
Model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


In [88]:
# Cell 6 — Generate user embeddings

user_texts = users["user_semantic_text"].fillna("").tolist()

user_embeddings = model.encode(
    user_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("User embeddings created.")
print("Shape:", user_embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

User embeddings created.
Shape: (50, 384)


In [89]:
# Cell 7 — Generate destination embeddings

destination_texts = destinations[
    "destination_semantic_text"
].fillna("").tolist()

destination_embeddings = model.encode(
    destination_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Destination embeddings created.")
print("Shape:", destination_embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Destination embeddings created.
Shape: (100, 384)


In [90]:
# Cell 8 — Verify embeddings

print("User embeddings:", user_embeddings.shape)
print("Destination embeddings:", destination_embeddings.shape)

assert user_embeddings.shape == (50, 384)
assert destination_embeddings.shape == (100, 384)

print("\nEmbedding dimensions are correct.")

User embeddings: (50, 384)
Destination embeddings: (100, 384)

Embedding dimensions are correct.


In [91]:
# Cell 9 — Calculate SBERT cosine similarity

from sklearn.metrics.pairwise import cosine_similarity

sbert_similarity_matrix = cosine_similarity(
    user_embeddings,
    destination_embeddings
)

print("SBERT similarity matrix created.")
print("Shape:", sbert_similarity_matrix.shape)
print("Minimum:", sbert_similarity_matrix.min())
print("Maximum:", sbert_similarity_matrix.max())
print("Mean:", sbert_similarity_matrix.mean())

SBERT similarity matrix created.
Shape: (50, 100)
Minimum: 0.22217414
Maximum: 0.7687525
Mean: 0.42915133


In [92]:
# Cell 10 — Create full SBERT results dataframe

sbert_rows = []

for user_idx, user_id in enumerate(users["user_id"]):
    for dest_idx, destination_id in enumerate(destinations["destination_id"]):
        sbert_rows.append({
            "user_id": user_id,
            "destination_id": destination_id,
            "sbert_similarity": sbert_similarity_matrix[
                user_idx,
                dest_idx
            ]
        })

sbert_results = pd.DataFrame(sbert_rows)

print("SBERT results created.")
print("Shape:", sbert_results.shape)
print("Columns:", sbert_results.columns.tolist())

SBERT results created.
Shape: (5000, 3)
Columns: ['user_id', 'destination_id', 'sbert_similarity']


In [93]:
# Cell 11 — Validate SBERT results

print("Rows:", len(sbert_results))
print("Users:", sbert_results["user_id"].nunique())
print("Destinations:", sbert_results["destination_id"].nunique())
print(
    "Duplicate user-destination pairs:",
    sbert_results.duplicated(
        subset=["user_id", "destination_id"]
    ).sum()
)

assert len(sbert_results) == 5000
assert sbert_results["user_id"].nunique() == 50
assert sbert_results["destination_id"].nunique() == 100
assert sbert_results.duplicated(
    subset=["user_id", "destination_id"]
).sum() == 0

print("\nSBERT pair validation PASSED.")

Rows: 5000
Users: 50
Destinations: 100
Duplicate user-destination pairs: 0

SBERT pair validation PASSED.


In [94]:
# Cell 12 — Normalize SBERT similarity

from sklearn.preprocessing import MinMaxScaler

sbert_scaler = MinMaxScaler()

sbert_results["sbert_similarity_normalized"] = (
    sbert_scaler.fit_transform(
        sbert_results[["sbert_similarity"]]
    ).ravel()
)

print(
    sbert_results[
        [
            "sbert_similarity",
            "sbert_similarity_normalized"
        ]
    ].describe()
)

       sbert_similarity  sbert_similarity_normalized
count       5000.000000                  5000.000000
mean           0.429151                     0.378678
std            0.078786                     0.144144
min            0.222174                     0.000000
25%            0.375996                     0.281427
50%            0.420537                     0.362917
75%            0.470305                     0.453971
max            0.768753                     1.000000


In [95]:
# Cell 13 — TF-IDF import

from sklearn.feature_extraction.text import TfidfVectorizer

print("TfidfVectorizer imported successfully.")

TfidfVectorizer imported successfully.


In [96]:
# Cell 14 — Prepare TF-IDF texts

users["tfidf_text"] = users["user_semantic_text"].fillna("")
destinations["tfidf_text"] = destinations[
    "destination_semantic_text"
].fillna("")

print("User TF-IDF texts:", len(users["tfidf_text"]))
print("Destination TF-IDF texts:", len(destinations["tfidf_text"]))

User TF-IDF texts: 50
Destination TF-IDF texts: 100


In [97]:
# Cell 15 — Build TF-IDF vectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1
)

combined_texts = (
    users["tfidf_text"].tolist()
    + destinations["tfidf_text"].tolist()
)

tfidf.fit(combined_texts)

print("TF-IDF vectorizer fitted successfully.")
print("Vocabulary size:", len(tfidf.vocabulary_))

TF-IDF vectorizer fitted successfully.
Vocabulary size: 1482


In [98]:
# Cell 16 — Transform texts into TF-IDF vectors

tfidf_user_vectors = tfidf.transform(
    users["tfidf_text"]
)

tfidf_destination_vectors = tfidf.transform(
    destinations["tfidf_text"]
)

print("User TF-IDF shape:", tfidf_user_vectors.shape)
print(
    "Destination TF-IDF shape:",
    tfidf_destination_vectors.shape
)

User TF-IDF shape: (50, 1482)
Destination TF-IDF shape: (100, 1482)


In [99]:
# Cell 17 — Calculate full TF-IDF similarity

tfidf_similarity_matrix = cosine_similarity(
    tfidf_user_vectors,
    tfidf_destination_vectors
)

print("TF-IDF similarity matrix created.")
print("Shape:", tfidf_similarity_matrix.shape)
print("Minimum:", tfidf_similarity_matrix.min())
print("Maximum:", tfidf_similarity_matrix.max())
print("Mean:", tfidf_similarity_matrix.mean())

TF-IDF similarity matrix created.
Shape: (50, 100)
Minimum: 0.008583769291935621
Maximum: 0.13871961180209086
Mean: 0.025350268730743267


In [100]:
# Cell 18 — Create full TF-IDF results dataframe

tfidf_rows = []

for user_idx, user_id in enumerate(users["user_id"]):
    for dest_idx, destination_id in enumerate(
        destinations["destination_id"]
    ):
        tfidf_rows.append({
            "user_id": user_id,
            "destination_id": destination_id,
            "content_similarity": tfidf_similarity_matrix[
                user_idx,
                dest_idx
            ]
        })

tfidf_full = pd.DataFrame(tfidf_rows)

print("Full TF-IDF results created.")
print("Shape:", tfidf_full.shape)
print("Columns:", tfidf_full.columns.tolist())

Full TF-IDF results created.
Shape: (5000, 3)
Columns: ['user_id', 'destination_id', 'content_similarity']


In [101]:
# Cell 19 — Normalize TF-IDF similarity

tfidf_scaler = MinMaxScaler()

tfidf_full["content_similarity_normalized"] = (
    tfidf_scaler.fit_transform(
        tfidf_full[["content_similarity"]]
    ).ravel()
)

print(
    tfidf_full[
        [
            "content_similarity",
            "content_similarity_normalized"
        ]
    ].describe()
)

       content_similarity  content_similarity_normalized
count         5000.000000                    5000.000000
mean             0.025350                       0.128838
std              0.020084                       0.154334
min              0.008584                       0.000000
25%              0.010483                       0.014591
50%              0.018469                       0.075961
75%              0.032919                       0.187002
max              0.138720                       1.000000


In [102]:
# Cell 20 — Context scoring helper functions

def normalize_text(value):
    return str(value).strip().lower()


def interest_match(user_interests, destination_interests):
    user_items = {
        normalize_text(x)
        for x in str(user_interests).split(";")
        if str(x).strip()
    }

    destination_items = {
        normalize_text(x)
        for x in str(destination_interests).split(";")
        if str(x).strip()
    }

    if not user_items or not destination_items:
        return 0.0

    matches = user_items.intersection(destination_items)

    return len(matches) / len(user_items)


def budget_fit(user_budget, destination_cost):
    try:
        user_budget = float(user_budget)
        destination_cost = float(destination_cost)

        if user_budget <= 0:
            return 0.0

        ratio = destination_cost / user_budget

        if ratio <= 1:
            return 1.0
        elif ratio <= 1.25:
            return 0.8
        elif ratio <= 1.5:
            return 0.5
        else:
            return 0.0

    except:
        return 0.0


def sustainability_fit(user_importance, destination_score):
    try:
        importance = normalize_text(user_importance)
        score = float(destination_score) / 100.0

        if importance in ["high", "very high"]:
            return score

        if importance == "medium":
            return 0.5 + (score * 0.5)

        return 0.5

    except:
        return 0.5

In [103]:
# Cell 21 — Build context scores for all user-destination pairs

context_rows = []

for _, user in users.iterrows():

    for _, destination in destinations.iterrows():

        interest_score = interest_match(
            user["preferred_interests"],
            destination["interest_tags"]
        )

        budget_score = budget_fit(
            user["budget_lkr_per_day"],
            destination["estimated_cost_lkr"]
        )

        sustainability_score = sustainability_fit(
            user["sustainability_importance"],
            destination["sustainability_score"]
        )

        context_score = (
            0.50 * interest_score
            + 0.30 * budget_score
            + 0.20 * sustainability_score
        )

        context_rows.append({
            "user_id": user["user_id"],
            "destination_id": destination["destination_id"],
            "interest_match": interest_score,
            "budget_fit": budget_score,
            "sustainability_fit": sustainability_score,
            "context_score": context_score
        })

all_context_scores = pd.DataFrame(context_rows)

print("Context scores created.")
print("Shape:", all_context_scores.shape)
print(
    "Columns:",
    all_context_scores.columns.tolist()
)

Context scores created.
Shape: (5000, 6)
Columns: ['user_id', 'destination_id', 'interest_match', 'budget_fit', 'sustainability_fit', 'context_score']


In [104]:
# Cell 22 — Validate context scores

print("Rows:", len(all_context_scores))
print(
    "Users:",
    all_context_scores["user_id"].nunique()
)
print(
    "Destinations:",
    all_context_scores["destination_id"].nunique()
)

print("\nContext score range:")
print(
    "Min:",
    all_context_scores["context_score"].min()
)

print(
    "Max:",
    all_context_scores["context_score"].max()
)

print(
    "Mean:",
    all_context_scores["context_score"].mean()
)

assert len(all_context_scores) == 5000
assert all_context_scores["user_id"].nunique() == 50
assert all_context_scores["destination_id"].nunique() == 100

print("\nContext validation PASSED.")

Rows: 5000
Users: 50
Destinations: 100

Context score range:
Min: 0.1
Max: 0.9
Mean: 0.54575

Context validation PASSED.


In [105]:
# Cell 23 — Merge all recommendation signals

recommendation_df = tfidf_full.merge(
    sbert_results,
    on=["user_id", "destination_id"],
    how="inner"
)

recommendation_df = recommendation_df.merge(
    all_context_scores,
    on=["user_id", "destination_id"],
    how="inner"
)

print("Merged recommendation dataframe.")
print("Shape:", recommendation_df.shape)
print("Columns:")
print(recommendation_df.columns.tolist())

Merged recommendation dataframe.
Shape: (5000, 10)
Columns:
['user_id', 'destination_id', 'content_similarity', 'content_similarity_normalized', 'sbert_similarity', 'sbert_similarity_normalized', 'interest_match', 'budget_fit', 'sustainability_fit', 'context_score']


In [106]:
# Cell 24 — Verify merged data

print("Rows:", len(recommendation_df))
print(
    "Unique users:",
    recommendation_df["user_id"].nunique()
)
print(
    "Unique destinations:",
    recommendation_df["destination_id"].nunique()
)

print(
    "Missing values:",
    recommendation_df.isna().sum().sum()
)

assert len(recommendation_df) == 5000
assert recommendation_df["user_id"].nunique() == 50
assert recommendation_df["destination_id"].nunique() == 100

print("\nMerge validation PASSED.")

Rows: 5000
Unique users: 50
Unique destinations: 100
Missing values: 0

Merge validation PASSED.


In [107]:
# Cell 25 — Calculate SBERT hybrid score

recommendation_df["sbert_hybrid_score"] = (
    0.40 * recommendation_df[
        "content_similarity_normalized"
    ]
    + 0.30 * recommendation_df[
        "sbert_similarity_normalized"
    ]
    + 0.30 * recommendation_df[
        "context_score"
    ]
)

print("SBERT hybrid score calculated.")

print(
    recommendation_df[
        ["sbert_hybrid_score"]
    ].describe()
)

SBERT hybrid score calculated.
       sbert_hybrid_score
count         5000.000000
mean             0.328864
std              0.123835
min              0.045112
25%              0.232375
50%              0.304059
75%              0.404706
max              0.804605


In [108]:
# Cell 26 — Add destination information

destination_columns = [
    "destination_id",
    "name",
    "category",
    "district",
    "estimated_cost_lkr",
    "duration_hours",
    "sustainability_score",
    "crowd_score",
    "seasonality_score"
]

destination_info = destinations[
    destination_columns
].copy()

recommendation_df = recommendation_df.merge(
    destination_info,
    on="destination_id",
    how="left"
)

print("Destination information added.")
print("Shape:", recommendation_df.shape)

Destination information added.
Shape: (5000, 19)


In [109]:
# Cell 27 — Rank all destinations for each user

recommendation_df = recommendation_df.sort_values(
    ["user_id", "sbert_hybrid_score"],
    ascending=[True, False]
).copy()

recommendation_df["rank"] = (
    recommendation_df
    .groupby("user_id")
    .cumcount()
    + 1
)

print("Ranking completed.")

print(
    recommendation_df[
        [
            "user_id",
            "destination_id",
            "sbert_hybrid_score",
            "rank"
        ]
    ].head(15).to_string(index=False)
)

Ranking completed.
user_id destination_id  sbert_hybrid_score  rank
   U001           D051            0.618589     1
   U001           D053            0.589435     2
   U001           D098            0.557703     3
   U001           D052            0.552463     4
   U001           D066            0.541864     5
   U001           D048            0.538383     6
   U001           D070            0.532182     7
   U001           D029            0.530365     8
   U001           D054            0.528928     9
   U001           D020            0.527826    10
   U001           D094            0.527488    11
   U001           D089            0.525252    12
   U001           D097            0.519232    13
   U001           D091            0.516677    14
   U001           D032            0.500480    15


In [110]:
# Cell 28 — Select top 10 recommendations per user

b5_sbert_recommendations = recommendation_df[
    recommendation_df["rank"] <= 10
].copy()

print("Top-10 recommendations created.")
print("Rows:", len(b5_sbert_recommendations))
print(
    "Users:",
    b5_sbert_recommendations["user_id"].nunique()
)
print(
    "Destinations:",
    b5_sbert_recommendations["destination_id"].nunique()
)

Top-10 recommendations created.
Rows: 500
Users: 50
Destinations: 77


In [111]:
# Cell 29 — Final B5 validation

print("=" * 60)
print("FINAL B5 SBERT RECOMMENDATION CHECK")
print("=" * 60)

print("Rows:", len(b5_sbert_recommendations))
print(
    "Users:",
    b5_sbert_recommendations["user_id"].nunique()
)
print(
    "Destinations:",
    b5_sbert_recommendations["destination_id"].nunique()
)

print(
    "Minimum rank:",
    b5_sbert_recommendations["rank"].min()
)

print(
    "Maximum rank:",
    b5_sbert_recommendations["rank"].max()
)

print(
    "Duplicate user-destination pairs:",
    b5_sbert_recommendations.duplicated(
        subset=["user_id", "destination_id"]
    ).sum()
)

print("\nRows per user:")
print(
    b5_sbert_recommendations
    .groupby("user_id")
    .size()
    .value_counts()
    .sort_index()
)

assert len(b5_sbert_recommendations) == 500
assert b5_sbert_recommendations["user_id"].nunique() == 50
assert b5_sbert_recommendations["rank"].min() == 1
assert b5_sbert_recommendations["rank"].max() == 10
assert (
    b5_sbert_recommendations
    .groupby("user_id")
    .size()
    .eq(10)
    .all()
)
assert (
    b5_sbert_recommendations
    .duplicated(
        subset=["user_id", "destination_id"]
    ).sum()
    == 0
)

print("\n" + "=" * 60)
print("B5 VALIDATION PASSED")
print("=" * 60)

FINAL B5 SBERT RECOMMENDATION CHECK
Rows: 500
Users: 50
Destinations: 77
Minimum rank: 1
Maximum rank: 10
Duplicate user-destination pairs: 0

Rows per user:
10    50
Name: count, dtype: int64

B5 VALIDATION PASSED


In [112]:
# Cell 30 — Inspect final recommendations

display(
    b5_sbert_recommendations[
        [
            "user_id",
            "destination_id",
            "name",
            "category",
            "district",
            "content_similarity_normalized",
            "sbert_similarity_normalized",
            "context_score",
            "sbert_hybrid_score",
            "rank"
        ]
    ].head(20)
)

,user_id,destination_id,name,category,district,content_similarity_normalized,sbert_similarity_normalized,context_score,sbert_hybrid_score,rank
50,U001,D051,Viharamahadevi Park,nature;city,colombo,0.384356,0.649489,0.90,0.618589,1
52,U001,D053,Dutch Hospital Colombo,heritage;city,colombo,0.492348,0.908321,0.40,0.589435,2
97,U001,D098,Ohiya,nature;adventure,badulla,0.423080,0.394902,0.90,0.557703,3
51,U001,D052,Independence Square,heritage;city,colombo,0.215321,0.904450,0.65,0.552463,4
65,U001,D066,Bopath Ella,nature,ratnapura,0.327315,0.469794,0.90,0.541864,5
47,U001,D048,Colombo National Museum,heritage;culture,colombo,0.333282,0.950233,0.40,0.538383,6
69,U001,D070,Brief Garden,nature,kalutara,0.311280,0.458900,0.90,0.532182,7
28,U001,D029,Ella Rock,nature;adventure,badulla,0.422302,0.304813,0.90,0.530365,8
53,U001,D054,Mount Lavinia Beach,beach,colombo,0.400792,0.578705,0.65,0.528928,9
19,U001,D020,Nuwara Eliya City,city;nature,nuwara eliya,0.441587,0.270637,0.90,0.527826,10


In [113]:
# Cell 31 — Inspect recommendations for one user

sample_user = b5_sbert_recommendations["user_id"].iloc[0]

print("Sample user:", sample_user)

display(
    b5_sbert_recommendations[
        b5_sbert_recommendations["user_id"] == sample_user
    ][
        [
            "user_id",
            "destination_id",
            "name",
            "category",
            "district",
            "sbert_hybrid_score",
            "rank"
        ]
    ].sort_values("rank")
)

Sample user: U001


,user_id,destination_id,name,category,district,sbert_hybrid_score,rank
50,U001,D051,Viharamahadevi Park,nature;city,colombo,0.618589,1
52,U001,D053,Dutch Hospital Colombo,heritage;city,colombo,0.589435,2
97,U001,D098,Ohiya,nature;adventure,badulla,0.557703,3
51,U001,D052,Independence Square,heritage;city,colombo,0.552463,4
65,U001,D066,Bopath Ella,nature,ratnapura,0.541864,5
47,U001,D048,Colombo National Museum,heritage;culture,colombo,0.538383,6
69,U001,D070,Brief Garden,nature,kalutara,0.532182,7
28,U001,D029,Ella Rock,nature;adventure,badulla,0.530365,8
53,U001,D054,Mount Lavinia Beach,beach,colombo,0.528928,9
19,U001,D020,Nuwara Eliya City,city;nature,nuwara eliya,0.527826,10


In [114]:
# Cell 32 — Save final B5 recommendations

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

b5_path = RESULTS_DIR / "b5_sbert_semantic_recommendations.csv"

b5_sbert_recommendations.to_csv(
    b5_path,
    index=False
)

print("B5 saved successfully.")
print("Path:", b5_path)
print("Rows:", len(b5_sbert_recommendations))

B5 saved successfully.
Path: ..\results\b5_sbert_semantic_recommendations.csv
Rows: 500


In [115]:
# Cell 33 — Save SBERT embeddings

import numpy as np

embeddings_dir = RESULTS_DIR / "embeddings"
embeddings_dir.mkdir(parents=True, exist_ok=True)

np.save(
    embeddings_dir / "user_sbert_embeddings.npy",
    user_embeddings
)

np.save(
    embeddings_dir / "destination_sbert_embeddings.npy",
    destination_embeddings
)

print("SBERT embeddings saved successfully.")
print("User embeddings:", embeddings_dir / "user_sbert_embeddings.npy")
print(
    "Destination embeddings:",
    embeddings_dir / "destination_sbert_embeddings.npy"
)

SBERT embeddings saved successfully.
User embeddings: ..\results\embeddings\user_sbert_embeddings.npy
Destination embeddings: ..\results\embeddings\destination_sbert_embeddings.npy


In [116]:
# Cell 34 — Final Notebook 04 verification

print("=" * 60)
print("NOTEBOOK 04 COMPLETE")
print("=" * 60)

print("\nB5 file:")
print(b5_path)
print("Exists:", b5_path.exists())

print("\nUser embeddings:")
user_embedding_path = (
    embeddings_dir / "user_sbert_embeddings.npy"
)
print(user_embedding_path)
print("Exists:", user_embedding_path.exists())

print("\nDestination embeddings:")
destination_embedding_path = (
    embeddings_dir / "destination_sbert_embeddings.npy"
)
print(destination_embedding_path)
print("Exists:", destination_embedding_path.exists())

print("\nFinal B5 shape:")
print(b5_sbert_recommendations.shape)

print("\nFinal B5 columns:")
print(b5_sbert_recommendations.columns.tolist())

print("\n" + "=" * 60)
print("04_sbert_semantic_recommendation.ipynb FINISHED")
print("=" * 60)

NOTEBOOK 04 COMPLETE

B5 file:
..\results\b5_sbert_semantic_recommendations.csv
Exists: True

User embeddings:
..\results\embeddings\user_sbert_embeddings.npy
Exists: True

Destination embeddings:
..\results\embeddings\destination_sbert_embeddings.npy
Exists: True

Final B5 shape:
(500, 20)

Final B5 columns:
['user_id', 'destination_id', 'content_similarity', 'content_similarity_normalized', 'sbert_similarity', 'sbert_similarity_normalized', 'interest_match', 'budget_fit', 'sustainability_fit', 'context_score', 'sbert_hybrid_score', 'name', 'category', 'district', 'estimated_cost_lkr', 'duration_hours', 'sustainability_score', 'crowd_score', 'seasonality_score', 'rank']

04_sbert_semantic_recommendation.ipynb FINISHED
